<div style="text-align: center;">
    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>
</div>

<div style="text-align: center;">
    <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3FsdzRveTBrenMxM3VnbDMwaTJxN2NnZm50aGFibXk1NzNnY2Q0MCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LR5ZBwZHv02lmpVoEU/giphy.gif" alt="NLP Gif" style="width: 300px; height: 150px; object-fit: cover; object-position: center;">
</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


<h2 style="color: #FF8C00;">PyPDFLoader</h2>

`PyPDFLoader` is a lightweight Python library designed to streamline the process of loading and parsing PDF documents for text processing tasks. It is particularly useful in Retrieval-Augmented Generation workflows where text extraction from PDFs is required.

- **What Does PyPDFLoader Do?**
  - Extracts text from PDF files, retaining formatting and layout.
  - Simplifies the preprocessing of document-based datasets.
  - Supports efficient and scalable loading of large PDF collections.

- **Key Features:**
  - Compatible with popular NLP libraries and frameworks.
  - Handles multi-page PDFs and embedded images (e.g., OCR-compatible setups).
  - Provides flexible configurations for structured text extraction.

- **Use Cases:**
  - Preparing PDF documents for retrieval-based systems in RAGs.
  - Automating the text extraction pipeline for document analysis.
  - Creating datasets from academic papers, technical manuals, and reports.


In [9]:
#pip install langchain langchain_community pypdf
#pip install termcolor langchain_openai langchain-huggingface sentence-transformers chromadb langchain_chroma tiktoken openai python-dotenv
#!pip install -U langchain langchain-community

In [13]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

import warnings
warnings.filterwarnings("ignore")


<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [14]:
# FILE PATH FOR THE DOCUMENT
file_path = "../ai-for-everyone.pdf"


<h3 style="color: #FF8C00;">Documents into pages</h3>

The `PyPDFLoader` library allows efficient loading and splitting of PDF documents into smaller, manageable parts for NLP tasks.

This functionality is particularly useful in workflows requiring granular text processing, such as Retrieval-Augmented Generation (RAG).


In [15]:
# LOAD AND SPLIT THE DOCUMENT
loader = PyPDFLoader(file_path)
pages = loader.load_and_split()
len(pages)


297

<h3 style="color: #FF8C00;">Pages into Chunks</h3>


####  RecursiveCharacterTextSplitter in LangChain

The `RecursiveCharacterTextSplitter` is the **recommended splitter** in LangChain when you want to break down long documents into smaller, semantically meaningful chunks — especially useful in **RAG pipelines**, where clean context chunks lead to better LLM responses.

####  Parameters

| Parameter       | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| `chunk_size`    | The **maximum number of characters** allowed in a chunk (e.g., `1000`).     |
| `chunk_overlap` | The number of **overlapping characters** between consecutive chunks (e.g., `200`). This helps preserve context continuity. |

####  How it works
`RecursiveCharacterTextSplitter` attempts to split the text **intelligently**, trying the following separators in order:
1. Paragraphs (`"\n\n"`)
2. Lines (`"\n"`)
3. Sentences or words (`" "`)
4. Individual characters (as a last resort)

This makes it ideal for handling **natural language documents**, such as PDFs, articles, or long reports, without breaking sentences or paragraphs in awkward ways.



In [16]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)
len(chunks)

1096

In [20]:
print(chunks[10:11])

[Document(metadata={'producer': 'Adobe PDF Library 16.0', 'creator': 'Adobe InDesign 16.4 (Macintosh)', 'creationdate': '2021-09-01T15:24:38+05:30', 'author': 'Pieter Verdegem', 'moddate': '2021-09-14T17:31:25-04:00', 'title': 'AI for Everyone?: Critical Perspectives', 'trapped': '/False', 'source': '../ai-for-everyone.pdf', 'total_pages': 310, 'page': 6, 'page_label': 'vi'}, page_content='Published By\nUniversity of Westminster Press\n115 New Cavendish Street\nLondon W1W 6UW\nwww.uwestminsterpress.co.uk\nIntroduction and Editorial Arrangement © Pieter Verdegem 2021\nText of Chapters © Several Contributors 2021\nFirst published 2021\nCover design: www.ketchup-productions.co.uk  \nSeries cover concept: Mina Bach (minabach.co.uk)\nPrint and digital versions typeset by Siliconchips Services Ltd. \nISBN (Paperback): 978-1-914386-16-9\nISBN (PDF): 978-1-914386-13-8\nISBN (EPUB): 978-1-914386-14-5\nISBN (Kindle): 978-914386-15-2\nDOI: https://doi.org/10.16997/book55\nThis work is licensed un

####  Alternative: CharacterTextSplitter

`CharacterTextSplitter` is a simpler splitter that breaks text into chunks based **purely on character count**, without trying to preserve any natural language structure.

##### Example:
```python
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
````

This method is faster and more predictable but may split text in the middle of a sentence or paragraph, which can hurt performance in downstream tasks like retrieval or QA.

---

#### Comparison Table

| Feature                        | RecursiveCharacterTextSplitter | CharacterTextSplitter     |
| ------------------------------ | ------------------------------ | ------------------------- |
| Structure-aware splitting      |  Yes                          |  No                      |
| Preserves sentence/paragraphs  |  Yes                          |  No                      |
| Risk of splitting mid-sentence |  Minimal                     |  High                   |
| Ideal for RAG/document QA      |  Highly recommended           |  Only if structured text |
| Performance speed              |  Slightly slower             |  Faster                  |

---

#### Recommendation

Use `RecursiveCharacterTextSplitter` for most real-world document processing tasks, especially when building RAG pipelines or working with structured natural language content like PDFs or articles.

## Best Practices for Choosing Chunk Size in RAG

### Best Practices for Chunk Size in RAG

| Factor                      | Recommendation                                                                                                                                                                                          |
| ---------------------------| ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **LLM context limit**       | Choose a chunk size that lets you retrieve multiple chunks **without exceeding the model’s token limit**. For example, GPT-4o supports 128k tokens, but with GPT-3.5 (16k) or GPT-4 (32k), keep it modest. |
| **Chunk size (in characters)** | Typically: **500–1,000 characters** per chunk → ~75–200 tokens. This fits well for retrieval + prompt without context overflow.                                                                           |
| **Chunk size (in tokens)**  | If using token-based splitter (e.g. `TokenTextSplitter`): aim for **100–300 tokens** per chunk.                                                                                                            |
| **Chunk overlap**           | Use **overlap of 10–30%** (e.g., 100–300 characters or ~50 tokens) to preserve context across chunk boundaries and avoid cutting off important ideas mid-sentence.                                        |
| **Document structure**      | Use **`RecursiveCharacterTextSplitter`** to preserve semantic boundaries (paragraphs, sentences) instead of arbitrary cuts.                                                                                |
| **Task type**               | For **question answering**, smaller chunks (~500–800 chars) reduce noise.<br>For **summarization**, slightly larger chunks (~1000–1500) are OK.                                                          |
| **Embedding model**         | Some models (e.g., `text-embedding-3-large`) can handle long input. But still, smaller chunks give **finer-grained retrieval**, which improves relevance.                                                  |
| **Query type**              | If users ask **very specific questions**, small focused chunks are better. For broader queries, bigger chunks might help.                                                                                  |


### Rule of Thumb

| Use Case                 | Chunk Size      | Overlap |
| ------------------------| --------------- | ------- |
| Factual Q&A              | 500–800 chars   | 100–200 |
| Summarization            | 1000–1500 chars | 200–300 |
| Technical documents      | 400–700 chars   | 100–200 |
| Long reports/books       | 800–1200 chars  | 200–300 |
| Small LLMs (≤16k tokens) | ≤800 chars      | 100–200 |


### Avoid

- Chunks >2000 characters: risks context overflow.
- No overlap: may lose key information between chunks.



<h2 style="color: #FF8C00;">Embeddings</h2>

Embeddings transform text into dense vector representations, capturing semantic meaning and contextual relationships. They are essential for efficient document retrieval and similarity analysis.

- **What are OpenAI Embeddings?**
  - Pre-trained embeddings like `text-embedding-3-large` generate high-quality vector representations for text.
  - Encapsulate semantic relationships in the text, enabling robust NLP applications.

- **Key Features of `text-embedding-3-large`:**
  - Large-scale embedding model optimized for accuracy and versatility.
  - Handles diverse NLP tasks, including retrieval, classification, and clustering.
  - Ideal for applications with high-performance requirements.

- **Benefits:**
  - Reduces the need for extensive custom training.
  - Provides state-of-the-art performance in retrieval-augmented systems.
  - Compatible with RAGs to create powerful context-aware models.


In [21]:
from langchain_community.embeddings import OpenAIEmbeddings
from dotenv import load_dotenv


In [25]:
load_dotenv(dotenv_path = "../../.env")

True

In [27]:
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")


<h2 style="color: #FF8C00;">ChromaDB</h2>

ChromaDB is a versatile vector database designed for efficiently storing and retrieving embeddings. It integrates seamlessly with embedding models to enable high-performance similarity search and context-based retrieval.

### Workflow Overview:
- **Step 1:** Generate embeddings using a pre-trained model (e.g., OpenAI's `text-embedding-3-large`).
- **Step 2:** Store the embeddings in ChromaDB for efficient retrieval and similarity calculations.
- **Step 3:** Use the stored embeddings to perform searches, matching, or context-based retrieval.

### Key Features of ChromaDB:
- **Scalability:** Handles large-scale datasets with optimized indexing and search capabilities.
- **Speed:** Provides fast and accurate retrieval of embeddings for real-time applications.
- **Integration:** Supports integration with popular frameworks and libraries for embedding generation.

In [32]:
from langchain_chroma import Chroma


In [34]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="/chroma_db_LAB")
print("ChromaDB created with document embeddings.")


ChromaDB created with document embeddings.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercice1: Write a user question that someone might ask about your book’s topic or content.

In [68]:
user_question = ["1. What are the main ways AI can be used in a business?", 
                 "2. How can AI create value in a business?",
                 "3. What are the main differences between AI and traditional software?",
                 "4. What skills does a non-technical person need to work with AI?",
                 "5. How should a company start building an AI strategy?",
                 "6. What are the most common misconceptions about artificial intelligence?",
                 "7. How can AI improve decision-making inside an organization?",
                 "8. What are the ethical risks associated with deploying AI systems?",
                 "9. How does machine learning differ from rule-based systems?",
                 "10. What types of business problems are best suited for AI solutions?",
                 "11. How should teams collaborate when building AI projects?"] # User question

results = []  # to store the results for each question

for question in user_question:
    retrieved_docs = db.similarity_search(question, k=10) # k is the number of documents to retrieve
    results.append(retrieved_docs)
    

In [70]:
results[0]

[Document(id='9e40f675-4859-424f-b615-7377f415706b', metadata={'creationdate': '2021-09-01T15:24:38+05:30', 'title': 'AI for Everyone?: Critical Perspectives', 'moddate': '2021-09-14T17:31:25-04:00', 'author': 'Pieter Verdegem', 'producer': 'Adobe PDF Library 16.0', 'page_label': '23', 'creator': 'Adobe InDesign 16.4 (Macintosh)', 'source': '../ai-for-everyone.pdf', 'page': 31, 'total_pages': 310, 'trapped': '/False'}, page_content='or cybernetics).\nArtificial intelligence can be classified into three types of systems: analyti -\ncal, human-inspired and humanised (Kaplan and Haenlein 2019). Analytical \nAI contains characteristics consistent with cognitive intelligence only: gener -\nating cognitive representation of the world and using learning based on past \nexperience to inform future decisions. Human-inspired AI contains elements \nof cognitive and emotional intelligence: understanding human emotions, \nin addition to cognitive elements, and considering them in their decision-  \

In [74]:
len(results)

11

In [80]:
# DISPLAY TOP RESULTS
for i, docs in enumerate(results) :
    print(f"\n=== QUESTION {i+1} ===")
    # Normalise the structure: always a list
    #docs_list = docs if isinstance(docs, list) else [docs]
    for j, item in enumerate(docs[:3]) :
        # If item is a tuple → the Document is item[0]
        #doc = item[0] if isinstance(item, tuple) else item
        print(f"\nDOCUMENT {j+1} : \n{item.page_content[36:1000]}\n")



=== QUESTION 1 ===

DOCUMENT 1 : 
ence can be classified into three types of systems: analyti -
cal, human-inspired and humanised (Kaplan and Haenlein 2019). Analytical 
AI contains characteristics consistent with cognitive intelligence only: gener -
ating cognitive representation of the world and using learning based on past 
experience to inform future decisions. Human-inspired AI contains elements 
of cognitive and emotional intelligence: understanding human emotions, 
in addition to cognitive elements, and considering them in their decision-  
making. Humanised AI contains characteristics of all types of competencies 
(i.e., cognitive, emotional and social intelligence), is able to be self-conscious, 
and is self-aware in interactions with others.
A robot driven by analytical artificial intelligence would be capable of 
answering queries concerning restaurant recommendations based on certain


DOCUMENT 2 : 
 and does not necessarily lead to a human 
being’s replacement. The Swedis

<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [83]:
def _get_document_prompt(docs):
    prompt = "\n"

    # Always normalise to a list
    #docs_list = docs if isinstance(docs, list) else [docs]

    for item in docs:

        # If item is a tuple → the Document is item[0]
        # doc = item[0] if isinstance(item, tuple) else item

        # Securing access to content
        content = getattr(item, "page_content", "")

        prompt += "\nCONTENT:\n"
        prompt += content + "\n\n"

    return prompt


In [86]:
# GENERATE A FORMATTED CONTEXT FROM THE RETRIEVED DOCUMENTS
formatted_contexts = []

for docs in results:
    formatted_contexts.append(_get_document_prompt(docs))

print("CONTEXTS FORMATTED FOR GPT MODEL.")


CONTEXTS FORMATTED FOR GPT MODEL.


<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [87]:
prompts = []

for question, context in zip(user_question, formatted_contexts):

    prompt = f"""
You are an AI assistant helping a business leader understand artificial intelligence.
Using only the information from the retrieved context, explain the answer in a clear,
non-technical, and practical way. Focus on business impact, strategy, and real-world
examples. If the context does not contain enough information, say so explicitly.

User question:
{question}

Context:
{context}

Answer:
"""
    prompts.append(prompt)


In [89]:
import openai

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [94]:
# SET UP GPT CLIENT AND PARAMETERS
client = openai.OpenAI()

model_params = {
    "model" : "gpt-4o",
    "temperature" : 0.2,       # LOW CREATIVITY => MORE FACTUAL AND GROUNDED
    "max_tokens" : 250,        # ALLOW FOR DETAILED ANSWERS
    "top_p" : 0.9,             # BALANCED NUCLEUS SAMPLING
    "frequency_penalty" : 0.2, # REDUCE REPETITION
    "presence_penalty" : 0.1   # SLIGHTLY ENCOURAGE NEW IDEAS
}

<h1 style="color: #FF6347;">Response</h1>


In [98]:
answers = []

for prompt in prompts :
    messages = [{"role" : "user", "content" : prompt}]
    completion = client.chat.completions.create(
        messages=messages,
        **model_params,
        timeout=120
    )
    answers.append(completion.choices[0].message.content)


In [100]:
for index, answer in enumerate(answers) :
    print(f"\n=== ANSWER TO QUESTION {index+1} ===\n")
    print(answer)

#answer = completion.choices[0].message.content
#print(answer)


=== ANSWER TO QUESTION 1 ===

The context provides several examples of how AI can be used in business, focusing on its practical applications and impact. Here are the main ways AI can be utilized:

1. **Customer Service Enhancement**: AI-driven virtual assistants, like SEB's AIDA, can handle a wide range of customer queries, such as making overseas payments or opening bank accounts. These systems can also detect customer emotions through voice tone and adapt their responses accordingly. This not only improves customer satisfaction but also frees up human employees to focus on more complex tasks.

2. **Recommendation Engines and Targeted Advertising**: AI can analyze customer data to provide personalized recommendations and targeted advertising, enhancing marketing strategies and increasing sales.

3. **Predictive Maintenance**: In industries like manufacturing, AI can predict equipment failures before they occur, reducing downtime and maintenance costs.

4. **Fraud Detection**: AI sys

In [105]:
len(answers[0])

1421

<img src="https://miro.medium.com/v2/resize:fit:824/1*GK56xmDIWtNQAD_jnBIt2g.png" alt="NLP Gif" style="width: 500px">

<h2 style="color: #FF6347;">Cosine Similarity</h2>

**Cosine similarity** is a metric used to measure the alignment or similarity between two vectors, calculated as the cosine of the angle between them. It is the **most common metric used in RAG pipelines** for vector retrieval.. It provides a scale from -1 to 1:

- **-1**: Vectors are completely opposite.
- **0**: Vectors are orthogonal (uncorrelated or unrelated).
- **1**: Vectors are identical.


<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg" alt="NLP Gif" style="width: 700px">

<h2 style="color: #FF6347;">Keyword Highlighting</h2>

Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [101]:
from termcolor import colored


The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [179]:
from termcolor import colored

def highlight_keywords(texts, keywords):
    highlighted_texts = []

    for text in texts:
        modified = text
        for keyword in keywords:
            modified = modified.replace(
                keyword,
                colored(keyword, "red", attrs=["bold"])
            )
        highlighted_texts.append(modified)

    return highlighted_texts


### Exercice4: add your keywords

In [181]:

query_keywords = ["AI", "machine learning", "data", "business", "strategy", "automation", "decision-making", "use case", "deployment", "organization"]
for i, docs in enumerate(results) :
    for j, doc in enumerate(docs[:1]):
        text = doc.page_content # RETRIVE THE TEXT
        snippet = text[:1000] # EXTRACT A SNIPPET
        highlighted = highlight_keywords([snippet], query_keywords) # HIGHLIGHT_KEYWORDS EXPECTS A LIST OF STRINGS
        print(f"Snippet {j+1}:\n{highlighted[0]}\n{'-'*80}")


Snippet 1:
or cybernetics).
Artificial intelligence can be classified into three types of systems: analyti -
cal, human-inspired and humanised (Kaplan and Haenlein 2019). Analytical 
AI contains characteristics consistent with cognitive intelligence only: gener -
ating cognitive representation of the world and using learning based on past 
experience to inform future decisions. Human-inspired AI contains elements 
of cognitive and emotional intelligence: understanding human emotions, 
in addition to cognitive elements, and considering them in their decision-  
making. Humanised AI contains characteristics of all types of competencies 
(i.e., cognitive, emotional and social intelligence), is able to be self-conscious, 
and is self-aware in interactions with others.
A robot driven by analytical artificial intelligence would be capable of 
answering queries concerning restaurant recommendations based on certain
------------------------------------------------------------------------------

1. `query_keywords` is a list of keywords to be highlighted.
2. The loop iterates over the first document in retrieved_docs.
3. For each document, a snippet of the first 200 characters is extracted.
4. The highlight_keywords function is called to highlight the keywords in the snippet.
5. The highlighted snippet is printed along with a separator line.

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:


In [146]:
# LOAD AND SPLIT THE DOCUMENT
loader_bonus = PyPDFLoader("../../../../../../BUSINESS/MARICEL BUSINESS/ENTREPRENDRE_DANS_LE_CHAOS/Entreprendre dans le Chaos - Manuscrit V1.pdf")
pages_bonus = loader_bonus.load_and_split()
len(pages_bonus)


445

In [147]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks_bonus = text_splitter.split_documents(pages_bonus)
len(chunks_bonus)

874

In [148]:
db_bonus = Chroma.from_documents(chunks_bonus, embeddings, persist_directory="../../chroma_db_LAB_bonus")
print("ChromaDB created with document embeddings.")


ChromaDB created with document embeddings.


In [158]:
user_question_bonus = ["Comment un entrepreneur peut-il transformer une période de chaos en opportunité stratégique ?",
                       "Quelles sont les qualités essentielles pour entreprendre dans un environnement instable ou imprévisible ?",
                       "Comment identifier les signaux faibles qui annoncent un changement majeur dans un marché ?",
                       "Quels types de décisions doivent être prises rapidement en période de crise, et lesquelles doivent être différées ?",
                       "Comment une entreprise peut-elle rester résiliente face à des perturbations économiques ou sociales ?",
                       "Quels sont les principaux risques auxquels un entrepreneur est confronté dans un contexte chaotique, et comment les gérer ?",
                       "Comment maintenir la motivation et la cohésion d’une équipe lorsque l’environnement externe devient incertain ?",
                       "Quelles stratégies permettent de naviguer efficacement dans un marché en transformation rapide ?",
                       "Comment un entrepreneur peut-il repenser son modèle économique pour survivre ou prospérer dans le chaos ?",
                       "Quel rôle joue l’innovation dans la capacité d’une entreprise à s’adapter à des conditions extrêmes ?",
                       "Comment développer une vision claire et mobilisatrice lorsque le futur est difficile à anticiper ?"] # User question

results_bonus = []  # to store the results for each question

for question in user_question_bonus:
    retrieved_docs_bonus = db_bonus.similarity_search(question, k=30) # k is the number of documents to retrieve
    results_bonus.append(retrieved_docs_bonus)
    

In [159]:
print(results_bonus[0])

[Document(id='4e7c3770-d6b6-4e3a-a564-bca1d9d66b03', metadata={'producer': 'Microsoft® Word pour Microsoft\xa0365', 'moddate': '2026-05-29T14:56:45+02:00', 'page': 412, 'creator': 'Microsoft® Word pour Microsoft\xa0365', 'total_pages': 446, 'creationdate': '2026-05-29T14:56:45+02:00', 'source': '../../../../../../BUSINESS/MARICEL BUSINESS/ENTREPRENDRE_DANS_LE_CHAOS/Entreprendre dans le Chaos - Manuscrit V1.pdf', 'page_label': '413'}, page_content='ENTREPRENDRE DANS LE CHAOS 1 E. CHRISTIAN BOSMANS \n \n411 \n \nDans le chaos, les systèmes rigides, lourds de bureaucratie ou \nd’une inertie forte, échouent. L’environnement les brise. Les sys-\ntèmes adaptatifs, eux, prospèrent. Leur souplesse structurelle est \nleur première arme concurrentielle. \nL’entrepreneur stratège ne gaspille pas son énergie à éradiquer \nle chaos. Une telle ambition relève de l’illusion. Son effort porte \nsur sa compréhension profonde et son utilisation. \nIl dresse une cartographie précise : \n● Les zones d’ine

In [160]:
# DISPLAY TOP RESULTS
for i, docs in enumerate(results_bonus) :
    print(f"\n=== QUESTION {i+1} ===")
    for j, item in enumerate(docs[:3]) :
        print(f"\nDOCUMENT {j+1} : \n{item.page_content[36:1000]}\n")



=== QUESTION 1 ===

DOCUMENT 1 : 
STIAN BOSMANS 
 
411 
 
Dans le chaos, les systèmes rigides, lourds de bureaucratie ou 
d’une inertie forte, échouent. L’environnement les brise. Les sys-
tèmes adaptatifs, eux, prospèrent. Leur souplesse structurelle est 
leur première arme concurrentielle. 
L’entrepreneur stratège ne gaspille pas son énergie à éradiquer 
le chaos. Une telle ambition relève de l’illusion. Son effort porte 
sur sa compréhension profonde et son utilisation. 
Il dresse une cartographie précise : 
● Les zones d’inefficience structurelle : Ce sont les failles 
du système (distribution lacunaire, services publics défi-
cients, infrastructures obsolètes). Pour l’entrepreneur, 
elles se muent en demande de solutions privées de subs-
titution. 
● Les vides institutionnels ou légaux : Là où l’État est ab-
sent ou la loi incertaine, une place existe pour celui qui 
propose un cadre clair ou un service de confiance. 
● Les besoins non couverts par le système formel : L’accès


D

In [161]:
# GENERATE A FORMATTED CONTEXT FROM THE RETRIEVED DOCUMENTS
formatted_contexts_bonus = []

for docs in results_bonus:
    formatted_contexts_bonus.append(_get_document_prompt(docs))

print("CONTEXTS FORMATTED FOR GPT MODEL.")


CONTEXTS FORMATTED FOR GPT MODEL.


In [162]:
prompts_bonus = []

for question, context in zip(user_question_bonus, formatted_contexts_bonus):

    prompt_bonus = f"""
    Tu es un assistant aidant un entrepreneur ou un dirigeant à comprendre comment 
    agir, décider et s’adapter dans des périodes de chaos, d’incertitude ou de transformation.
    En t’appuyant uniquement sur les informations du contexte fourni, explique la réponse 
    de manière claire, pratique et orientée terrain. Mets l’accent sur la résilience, 
    la prise de décision, la stratégie adaptative, le leadership et les opportunités 
    qui émergent dans l’instabilité. Si le contexte ne contient pas assez d’informations, 
    dis-le explicitement.
    
    Question de l’utilisateur :
    {question}
    
    Contexte :
    {context}
    
    Réponse :
    """
    prompts_bonus.append(prompt_bonus)


In [172]:
model_params_bonus = {
    "model" : "gpt-4o",
    "temperature" : 0.2,       # LOW CREATIVITY => MORE FACTUAL AND GROUNDED
    "max_tokens" : 1000,        # ALLOW FOR DETAILED ANSWERS
    "top_p" : 0.9,             # BALANCED NUCLEUS SAMPLING
    "frequency_penalty" : 0.2, # REDUCE REPETITION
    "presence_penalty" : 0.1   # SLIGHTLY ENCOURAGE NEW IDEAS
}

In [ ]:
answers_bonus = []

for prompt in prompts_bonus :
    messages = [{"role" : "user", "content" : prompt}]
    completion = client.chat.completions.create(
        messages=messages,
        **model_params_bonus,
        timeout=120
    )
    answers_bonus.append(completion.choices[0].message.content)


In [174]:
for index, answer in enumerate(answers_bonus) :
    print(f"\n=== ANSWER TO QUESTION {index+1} ===\n{user_question_bonus[index]}\n")
    print(answer, "\n\n")



=== ANSWER TO QUESTION 1 ===
Comment un entrepreneur peut-il transformer une période de chaos en opportunité stratégique ?

Pour transformer une période de chaos en opportunité stratégique, un entrepreneur doit adopter une approche proactive et adaptative, en tirant parti des caractéristiques uniques de l'environnement chaotique. Voici quelques stratégies pratiques basées sur le contexte fourni :

1. **Adopter une mentalité anti-fragile** : Plutôt que de simplement résister au chaos, un entrepreneur doit chercher à en tirer parti. Les entreprises anti-fragiles prospèrent dans l'incertitude en utilisant les chocs et le stress pour se renforcer. Cela implique de voir le chaos non pas comme un obstacle, mais comme une opportunité de croissance et d'innovation.

2. **Cartographier les inefficiences et les vides** : Identifier les zones d'inefficience structurelle, les vides institutionnels ou légaux, et les besoins non couverts par le système formel peut révéler des opportunités pour offr

In [183]:
query_keywords_bonus = [
    # Thèmes centraux
    "chaos", "incertitude", "résilience", "opportunité", "crise",
    "stratégie", "adaptation", "leadership", "risque", "transformation",
    "signal faible", "modèle économique", "innovation",

    # Ajout précédent
    "vision", "agilité", "perturbation", "changement", "conflit",
    "décision", "priorité", "organisation", "stress", "complexité",
    "survie", "pivot", "reconstruction",

    # Nouveaux mots (doublement enrichis)
    "anticipation", "intuition", "résolution", "effondrement", "rebond",
    "opportunisme", "résistance", "solidarité", "communication", "mobilisation",
    "alignement", "cohésion", "urgence", "fragilité", "stabilité",
    "vision long terme", "réinvention", "réorientation", "analyse",
    "diagnostic", "proactivité", "réactivité", "turbulence", "désordre",
    "systémique", "efficacité", "priorisation", "réallocation", "capacité",
    "endurance", "persévérance", "audace", "initiative", "créativité",
    "rupture", "transition", "incidence", "impact", "résultats"
]


for i, docs in enumerate(results_bonus) :
    for j, doc in enumerate(docs[:1]):
        text = doc.page_content # RETRIVE THE TEXT
        snippet = text[:1000] # EXTRACT A SNIPPET
        highlighted = highlight_keywords([snippet], query_keywords_bonus) # HIGHLIGHT_KEYWORDS EXPECTS A LIST OF STRINGS
        print(f"Snippet {j+1}:\n{highlighted[0]}\n{'-'*80}")


Snippet 1:
ENTREPRENDRE DANS LE CHAOS 1 E. CHRISTIAN BOSMANS 
 
411 
 
Dans le chaos, les systèmes rigides, lourds de bureaucratie ou 
d’une inertie forte, échouent. L’environnement les brise. Les sys-
tèmes adaptatifs, eux, prospèrent. Leur souplesse structurelle est 
leur première arme concurrentielle. 
L’entrepreneur stratège ne gaspille pas son énergie à éradiquer 
le chaos. Une telle ambition relève de l’illusion. Son effort porte 
sur sa compréhension profonde et son utilisation. 
Il dresse une cartographie précise : 
● Les zones d’inefficience structurelle : Ce sont les failles 
du système (distribution lacunaire, services publics défi-
cients, infrastructures obsolètes). Pour l’entrepreneur, 
elles se muent en demande de solutions privées de subs-
titution. 
● Les vides institutionnels ou légaux : Là où l’État est ab-
sent ou la loi incertaine, une place existe pour celui qui 
propose un cadre clair ou un service de confiance. 
● Les besoins non couverts par le système formel :